# 5. Visualising the source estimates

Notebook 04 computed a minimum-norm solution and saved it as `.stc` files. Here we look at
the result: on the cortical surface, as a time course, and averaged within anatomical
parcels.

<div class="alert alert-success">
    <b>Learning Objectives</b>:
     <ul>
      <li>Get inspired on how to visualize your computed source estimates! </li>
      <li>Color bar handling for zero centered and positive only values. </li>
    </ul>
</div>

In [ ]:
# imports
import os
import mne
import numpy as np
import matplotlib.pyplot as plt

# render the 3D brains inline in JupyterLab
mne.viz.set_3d_backend("notebook")

# data paths - same prefix as in notebook 04
subID = 24
data_path = "data"
subject_path = os.path.join(f"sub-0{subID}", "ses-mecha", "eeg")
base = os.path.join("..", data_path, subject_path, f"sub-0{subID}_ses-mecha_task-NT_")

# fsaverage: downloads it on first use, otherwise just returns the path
fs_dir = mne.datasets.fetch_fsaverage()
subjects_dir = os.path.dirname(fs_dir)
print(subjects_dir)

## Morphing to fsaverage

The source estimates live in the individual source space of this participant. We cannot plot
them on that brain: the anatomical MRI and the FreeSurfer surfaces of our participants are
not part of this dataset, for privacy reasons.

Instead we use the **morph** that ships with the data. It maps the individual cortical
surface onto *fsaverage*, the FreeSurfer average brain, by aligning the folding pattern.
This is also the step you would need for a group analysis: only after morphing are vertex
numbers comparable across participants.

In [ ]:
# load the source estimate and the precomputed morph
stc = mne.read_source_estimate(base + "MNE-all")
morph = mne.read_source_morph(base + "morph.h5")

print(stc)
print(morph)

In [ ]:
# apply the morph: individual source space -> fsaverage
stc_fs = morph.apply(stc)
print(stc_fs)

## The brain plot

`stc.plot()` paints the estimate on the cortical surface. The most important arguments:

- `surface="inflated"` - unfolds the cortex so that activity inside the sulci stays visible.
- `hemi="both"` - both hemispheres (`"lh"`, `"rh"` or `"split"` also work).
- `initial_time` - which time point to show when the figure opens.
- `time_viewer=True` - adds a time slider, so you can step through the epoch or play it as
  a movie.

Remember from notebook 04 that our solution is fixed-orientation, so the values are
**signed**: red and blue mean current flowing out of and into the cortical surface, not
"more" and "less" activity.

In [ ]:
# when and where is the estimate largest?
peak_vertex, peak_time = stc_fs.get_peak(tmin=0.05, tmax=0.4)
print(f"peak at {peak_time * 1000:.0f} ms")

In [ ]:
brain = stc_fs.plot(
    subject="fsaverage",
    subjects_dir=subjects_dir,
    surface="inflated",
    hemi="both",
    initial_time=peak_time,
    time_viewer=True,
)

### Colour scale

The plot above used the default colour scale. `clim` lets you set it yourself:
`kind="percent"` puts the three thresholds (transparent / saturating / maximum) at
percentiles of the data, so raising them shows fewer, more focal sources. Because our
estimates are signed we pass `pos_lims`, which mirrors the same limits to the negative side.
`kind="value"` takes absolute numbers instead - that is what you need when several
conditions have to share one scale.

In [ ]:
brain = stc_fs.plot(
    subject="fsaverage",
    subjects_dir=subjects_dir,
    surface="inflated",
    hemi="both",
    initial_time=peak_time,
    clim=dict(kind="percent", pos_lims=[97, 99, 99.95]),
    time_viewer=True,
)

## Time course of a single source

Every row of `stc.data` is the estimated time course of one vertex. `get_peak` with
`vert_as_index=True` gives us the row index of the strongest source, so we can plot it
directly.

In [ ]:
peak_idx, peak_time = stc_fs.get_peak(tmin=0.05, tmax=0.4, vert_as_index=True)
n_lh = len(stc_fs.vertices[0])
hemi = "lh" if peak_idx < n_lh else "rh"
print(f"strongest source: row {peak_idx} ({hemi}), peaking at {peak_time * 1000:.0f} ms")

plt.plot(stc_fs.times, stc_fs.data[peak_idx])
plt.axvline(0, color="k", linestyle="--")
plt.axvline(peak_time, color="r", linestyle=":")
plt.xlabel("time (s)")
plt.ylabel("amplitude (Am)")
plt.title(f"peak source ({hemi})")

## Anatomical parcels

Instead of single vertices we can average within anatomical regions. `aparc` is the
Desikan-Killiany parcellation that comes with FreeSurfer: 68 cortical labels, defined on
fsaverage and therefore directly usable on our morphed estimate.

`mode="mean_flip"` flips the sign of vertices whose surface normal points the other way
before averaging - without it, neighbouring sources with opposite orientation would cancel.

Keep §3.4 in mind: because of source leakage, a label time course is **not** "the activity
of that region". It is a weighted mixture that happens to be centred there.

In [ ]:
# the parcellation and the source space it is defined on
labels = mne.read_labels_from_annot("fsaverage", parc="aparc", subjects_dir=subjects_dir)
src_fs = mne.read_source_spaces(
    os.path.join(subjects_dir, "fsaverage", "bem", "fsaverage-ico-5-src.fif")
)
print(f"{len(labels)} labels, e.g.", [label.name for label in labels[:5]])

In [ ]:
# a few regions that are plausible for a tactile detection task
roi_names = ["postcentral-lh", "postcentral-rh", "superiorparietal-lh", "precuneus-lh"]
rois = [label for label in labels if label.name in roi_names]

label_tc = mne.extract_label_time_course(stc_fs, rois, src_fs, mode="mean_flip")
print("label time courses:", label_tc.shape)

In [ ]:
for name, time_course in zip([label.name for label in rois], label_tc):
    plt.plot(stc_fs.times, time_course, label=name)
plt.axvline(0, color="k", linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("amplitude (Am)")
plt.legend()
plt.title("label time courses (mean_flip)")

In [ ]:
# where these labels sit on the brain
brain = stc_fs.plot(
    subject="fsaverage",
    subjects_dir=subjects_dir,
    surface="inflated",
    hemi="both",
    initial_time=peak_time,
    time_viewer=False,
)
for label in rois:
    brain.add_label(label, borders=True)

## The contrast

Finally the difference between the two conditions - the perceived minus unperceived contrast
we looked at on the sensor level in notebook 02, now in source space.

In [ ]:
stc_difference = mne.read_source_estimate(base + "MNE-difference")
stc_difference_fs = morph.apply(stc_difference)

diff_vertex, diff_time = stc_difference_fs.get_peak(tmin=0.05, tmax=0.4)
print(f"largest difference at {diff_time * 1000:.0f} ms")

brain = stc_difference_fs.plot(
    subject="fsaverage",
    subjects_dir=subjects_dir,
    surface="inflated",
    hemi="both",
    initial_time=diff_time,
    clim=dict(kind="percent", pos_lims=[97, 99, 99.95]),
    time_viewer=True,
)